In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("../rqs/random_selection")

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "TensorFlow",
    "microsoft_vscode": "VsCode",
}

MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]


F1_COL = "macro_f1"

# tolerance for considering two F1 values equal
F1_TOL = 0.0005


# ============================================================
# LOAD DATA
# ============================================================

rows = []

for repo_dir in BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():

        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required = {"FewShot_Count", F1_COL, "Token_Count"}

            if not required.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
            df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", F1_COL, "Token_Count"])

            for _, r in df.iterrows():

                rows.append({
                    "Repository": repo,
                    "Model": model_dir.name,
                    "FewShot_Count": int(r["FewShot_Count"]),
                    "F1": float(r[F1_COL]),
                    "Tokens": int(r["Token_Count"]),
                })


data = pd.DataFrame(rows)

if data.empty:
    raise RuntimeError(
        "No data loaded. Check CSV files and columns."
    )

# ============================================================
# GROUP BY SIMILAR PERFORMANCE
# ============================================================

data["F1Bucket"] = (data["F1"] / F1_TOL).round() * F1_TOL

grouped = (
    data
    .groupby(["Repository", "Model", "FewShot_Count", "F1Bucket"])
    .agg(
        Best_F1=("F1", "max"),
        Min_Tokens=("Tokens", "min"),
        Max_Tokens=("Tokens", "max"),
        Count=("Tokens", "count"),
    )
    .reset_index()
)

# keep only cases where multiple prompts give same performance
grouped = grouped[grouped["Count"] > 1]


# ============================================================
# TOKEN VARIABILITY METRICS
# ============================================================

grouped["ΔTokens"] = grouped["Max_Tokens"] - grouped["Min_Tokens"]

grouped["Token_Savings_%"] = (
    grouped["ΔTokens"] / grouped["Max_Tokens"] * 100
)

# ============================================================
# SELECT MOST INTERESTING CASE PER MODEL
# ============================================================

summary_rows = []

for (repo, model), g in grouped.groupby(["Repository", "Model"]):

    if g.empty:
        continue

    # pick highest F1 first
    max_f1 = g["Best_F1"].max()

    g_high = g[g["Best_F1"] == max_f1]

    # among those choose largest token gap
    row = g_high.loc[g_high["ΔTokens"].idxmax()]

    summary_rows.append({

        "Repository": REPO_NAME_MAP.get(repo, repo),
        "Model": model,
        "FewShots": int(row["FewShot_Count"]),
        "F1": round(row["Best_F1"], 3),
        "MinTokens": int(row["Min_Tokens"]),
        "MaxTokens": int(row["Max_Tokens"]),
        "ΔTokens": int(row["ΔTokens"]),
        "Savings_%": round(row["Token_Savings_%"], 1),

    })

summary = pd.DataFrame(summary_rows)

summary = summary[summary["Model"].isin(MODELS)]
summary = summary.sort_values(["Repository", "Model"])

summary

,Repository,Model,FewShots,F1,MinTokens,MaxTokens,ΔTokens,Savings_%
0,Bitcoin,Llama-3.1-8B-Instruct,10,0.557,2665,3342,677,20.3
2,Bitcoin,Mistral-7B-Instruct-v0.3,10,0.696,10541,10616,75,0.7
3,Bitcoin,Qwen2.5-7B-Instruct,20,0.805,6540,16324,9784,59.9
4,Bitcoin,gemma-3-12b-it,6,0.801,1481,1805,324,18.0
15,OpenCV,Llama-3.1-8B-Instruct,1,0.274,175,197,22,11.2
17,OpenCV,Mistral-7B-Instruct-v0.3,20,0.414,13288,13614,326,2.4
18,OpenCV,Qwen2.5-7B-Instruct,32,0.711,10946,12023,1077,9.0
19,OpenCV,gemma-3-12b-it,24,0.699,10280,12640,2360,18.7
5,React,Llama-3.1-8B-Instruct,4,0.516,561,1641,1080,65.8
7,React,Mistral-7B-Instruct-v0.3,24,0.609,6011,6130,119,1.9


In [7]:
# ============================================================
# STEP 4: BUILD LATEX TABLE
# ============================================================

latex_lines = []

for repo, repo_df in summary.groupby("Repository", sort=False):

    repo_df = repo_df.reset_index(drop=True)

    for i, r in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
            if i == 0 else ""
        )

        latex_lines.append(
            f"            {repo_cell} & {r['Model']} & "
            f"{r['F1']:.2f} & {r['FewShots']} & "
            f"{r['MinTokens']} & {r['MaxTokens']} & "
            f"\\textbf{{{r['ΔTokens']}}} & "
            f"{r['Savings_%']:.1f}\\% \\\\"
        )

    latex_lines.append("            \\midrule")

latex_lines = latex_lines[:-1]

latex = "\n".join([
    r"\begin{table*}[t]",
    r"    \centering",
    r"    \caption{Token variability across prompts with identical macro-F1 and few-shot count, "
    r"computed per repository and model using stratified few-shot selection.}",
    r"    \label{tab:token-var-stratified-per-model}",
    r"    \resizebox{\textwidth}{!}{",
    r"        \begin{tabular}{llcccccc}",
    r"            \toprule",
    r"            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{\# FS} & "
    r"\textbf{Min Tokens} & \textbf{Max Tokens} & $\Delta$\textbf{Tokens} & \textbf{Token Savings} \\",
    r"            \midrule",
    *latex_lines,
    r"            \bottomrule",
    r"        \end{tabular}",
    r"    }",
    r"\end{table*}",
])


# ============================================================
# PRINT CLEAN LATEX
# ============================================================

for line in latex.splitlines():
    print(line)

\begin{table*}[t]
    \centering
    \caption{Token variability across prompts with identical macro-F1 and few-shot count, computed per repository and model using stratified few-shot selection.}
    \label{tab:token-var-stratified-per-model}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llcccccc}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{\# FS} & \textbf{Min Tokens} & \textbf{Max Tokens} & $\Delta$\textbf{Tokens} & \textbf{Token Savings} \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.56 & 10 & 2665 & 3342 & \textbf{677} & 20.3\% \\
             & Mistral-7B-Instruct-v0.3 & 0.70 & 10 & 10541 & 10616 & \textbf{75} & 0.7\% \\
             & Qwen2.5-7B-Instruct & 0.81 & 20 & 6540 & 16324 & \textbf{9784} & 59.9\% \\
             & gemma-3-12b-it & 0.80 & 6 & 1481 & 1805 & \textbf{324} & 18.0\% \\
            \midrule
            \multirow{4}{*}{OpenCV} & Llama-3.1-8B-Instruct & 0.27 & 1 & 

In [ ]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

# ============================================================
# CONFIG
# ============================================================

BASE_DIR    = Path("../rqs/random_selection")
EXP_NUM     = "random_selection"

REPO_NAME_MAP = {
    "facebook_react":        "React",
    "bitcoin_bitcoin":       "Bitcoin",
    "opencv_opencv":         "OpenCV",
    "tensorflow_tensorflow": "TensorFlow",
    "microsoft_vscode":      "VsCode",
}

MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

F1_COL  = "Macro_F1"
F1_TOL  = 0.0005

# ============================================================
# LOAD cl100k TOKEN COUNTS
# — one count per (repo, k, trial_index), model-agnostic
# ============================================================

cl100k_lookup = {}  # (repo, k, trial) -> cl100k_tokens

for repo in REPO_NAME_MAP.keys():
    cl100k_file = BASE_DIR / repo / "cl100k_token_counts.json"
    if not cl100k_file.exists():
        print(f"[MISSING cl100k] {cl100k_file}")
        continue
    with open(cl100k_file, "r") as f:
        data = json.load(f)
    for k_str, trials in data.items():
        k = int(k_str)
        for trial in trials:
            cl100k_lookup[(repo, k, trial["trial"])] = trial["cl100k_tokens"]

# ============================================================
# LOAD NATIVE TOKEN DATA
# ============================================================

rows = []

for repo_dir in BASE_DIR.iterdir():
    if not repo_dir.is_dir():
        continue
    repo      = repo_dir.name
    models_dir = repo_dir / "models"
    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue
        for model_dir in provider_dir.iterdir():
            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"
            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required = {"FewShot_Count", F1_COL, "Token_Count", "Trial"}
            if not required.issubset(df.columns):
                print(f"[MISSING COLS] {csv_path} — has: {list(df.columns)}")
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df[F1_COL]          = pd.to_numeric(df[F1_COL],          errors="coerce")
            df["Token_Count"]   = pd.to_numeric(df["Token_Count"],   errors="coerce")
            df["Trial"]         = pd.to_numeric(df["Trial"],         errors="coerce")

            df = df.dropna(subset=["FewShot_Count", F1_COL, "Token_Count", "Trial"])

            for _, r in df.iterrows():
                k           = int(r["FewShot_Count"])
                trial_idx   = int(r["Trial"])
                cl100k_tok  = cl100k_lookup.get((repo, k, trial_idx))

                rows.append({
                    "Repository":   repo,
                    "Model":        model_dir.name,
                    "FewShot_Count": k,
                    "Trial":        trial_idx,
                    "F1":           float(r[F1_COL]),
                    "Tokens":       int(r["Token_Count"]),
                    "cl100k_Tokens": cl100k_tok,
                })

data = pd.DataFrame(rows)

if data.empty:
    raise RuntimeError("No data loaded. Check CSV files and columns.")

# ============================================================
# GROUP BY SIMILAR PERFORMANCE
# ============================================================

data["F1Bucket"] = (data["F1"] / F1_TOL).round() * F1_TOL

grouped = (
    data
    .groupby(["Repository", "Model", "FewShot_Count", "F1Bucket"])
    .agg(
        Best_F1       = ("F1",            "max"),
        Min_Tokens    = ("Tokens",        "min"),
        Max_Tokens    = ("Tokens",        "max"),
        Min_cl100k    = ("cl100k_Tokens", "min"),
        Max_cl100k    = ("cl100k_Tokens", "max"),
        Count         = ("Tokens",        "count"),
    )
    .reset_index()
)

# keep only cases where multiple prompts give same performance
grouped = grouped[grouped["Count"] > 1]

# ============================================================
# TOKEN VARIABILITY METRICS — native
# ============================================================

grouped["ΔTokens"]         = grouped["Max_Tokens"] - grouped["Min_Tokens"]
grouped["Token_Savings_%"] = grouped["ΔTokens"] / grouped["Max_Tokens"] * 100

# ============================================================
# TOKEN VARIABILITY METRICS — cl100k
# ============================================================

grouped["Δcl100k"]           = grouped["Max_cl100k"] - grouped["Min_cl100k"]
grouped["cl100k_Savings_%"]  = grouped["Δcl100k"] / grouped["Max_cl100k"] * 100

# ============================================================
# SELECT MOST INTERESTING CASE PER (REPO, MODEL)
# ============================================================

summary_rows = []

for (repo, model), g in grouped.groupby(["Repository", "Model"]):
    if g.empty:
        continue

    max_f1  = g["Best_F1"].max()
    g_high  = g[g["Best_F1"] == max_f1]
    row     = g_high.loc[g_high["ΔTokens"].idxmax()]

    summary_rows.append({
        "Repository":      REPO_NAME_MAP.get(repo, repo),
        "Model":           model,
        "FewShots":        int(row["FewShot_Count"]),
        "F1":              round(row["Best_F1"], 3),
        # native
        "Min Tokens":      int(row["Min_Tokens"]),
        "Max Tokens":      int(row["Max_Tokens"]),
        "ΔTokens":         int(row["ΔTokens"]),
        "Savings% (native)": round(row["Token_Savings_%"], 1),
        # cl100k
        "Min cl100k":      int(row["Min_cl100k"])    if pd.notna(row["Min_cl100k"]) else None,
        "Max cl100k":      int(row["Max_cl100k"])    if pd.notna(row["Max_cl100k"]) else None,
        "Δcl100k":         int(row["Δcl100k"])       if pd.notna(row["Δcl100k"])    else None,
        "Savings% (cl100k)": round(row["cl100k_Savings_%"], 1) if pd.notna(row["cl100k_Savings_%"]) else None,
    })

summary = pd.DataFrame(summary_rows)
summary = summary[summary["Model"].isin(MODELS)]
summary = summary.sort_values(["Repository", "Model"])

summary

,Repository,Model,FewShots,F1,Min Tokens,Max Tokens,ΔTokens,Savings% (native),Min cl100k,Max cl100k,Δcl100k,Savings% (cl100k)
0,Bitcoin,Llama-3.1-8B-Instruct,10,0.557,2665,3342,677,20.3,2665,3342,677,20.3
2,Bitcoin,Mistral-7B-Instruct-v0.3,10,0.696,10541,10616,75,0.7,7577,7922,345,4.4
3,Bitcoin,Qwen2.5-7B-Instruct,20,0.805,6540,16324,9784,59.9,5645,14950,9305,62.2
4,Bitcoin,gemma-3-12b-it,6,0.801,1481,1805,324,18.0,1286,1505,219,14.6
15,OpenCV,Llama-3.1-8B-Instruct,1,0.274,175,197,22,11.2,175,197,22,11.2
17,OpenCV,Mistral-7B-Instruct-v0.3,20,0.414,13288,13614,326,2.4,10063,10647,584,5.5
18,OpenCV,Qwen2.5-7B-Instruct,32,0.711,10946,12023,1077,9.0,10243,11479,1236,10.8
19,OpenCV,gemma-3-12b-it,24,0.699,10280,12640,2360,18.7,8753,10535,1782,16.9
5,React,Llama-3.1-8B-Instruct,4,0.516,561,1641,1080,65.8,561,1641,1080,65.8
7,React,Mistral-7B-Instruct-v0.3,24,0.609,6011,6130,119,1.9,4968,5073,105,2.1


In [10]:
# ============================================================
# STEP 4: BUILD LATEX TABLE (native + cl100k)
# ============================================================

latex_lines = []

for repo, repo_df in summary.groupby("Repository", sort=False):
    repo_df = repo_df.reset_index(drop=True)
    for i, r in repo_df.iterrows():
        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
            if i == 0 else ""
        )
        latex_lines.append(
            f"            {repo_cell} & {r['Model']} & "
            f"{r['F1']:.2f} & {r['FewShots']} & "
            f"{int(r['Min Tokens']):,} & {int(r['Max Tokens']):,} & "
            f"{int(r['ΔTokens']):,} & "
            f"{r['Savings% (native)']:.1f}\\% & "
            f"{int(r['Min cl100k']):,} & {int(r['Max cl100k']):,} & "
            f"{int(r['Δcl100k']):,} & "
            f"{r['Savings% (cl100k)']:.1f}\\% \\\\"
        )
    latex_lines.append("            \\midrule")

latex_lines = latex_lines[:-1]

latex = "\n".join([
    r"\begin{table*}[t]",
    r"    \centering",
    r"    \caption{Token variability across prompts with identical macro-F1 "
    r"and few-shot count, computed per repository and model. "
    r"Native token counts (left) and \texttt{cl100k}-normalized token counts "
    r"(right) are reported alongside each other. Savings percentages are "
    r"consistent across both tokenizers, confirming that the reported "
    r"variability reflects genuine differences in prompt content.}",
    r"    \label{tab:token-range-random-selection-sim_k_sim_fs_sim_f1}",
    r"    \resizebox{\textwidth}{!}{",
    r"        \begin{tabular}{llcc"
    r"  ccc c"   # native
    r"  ccc c}",  # cl100k
    r"            \toprule",
    r"            & & & & "
    r"\multicolumn{4}{c}{\textbf{Native Tokens}} & "
    r"\multicolumn{4}{c}{\textbf{cl100k Tokens}} \\",
    r"            \cmidrule(lr){5-8} \cmidrule(lr){9-12}",
    r"            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{\# FS} & "
    r"\textbf{Min} & \textbf{Max} & $\Delta$ & \textbf{Savings} & "
    r"\textbf{Min} & \textbf{Max} & $\Delta$ & \textbf{Savings} \\",
    r"            \midrule",
    *latex_lines,
    r"            \bottomrule",
    r"        \end{tabular}",
    r"    }",
    r"\end{table*}",
])

for line in latex.splitlines():
    print(line)

\begin{table*}[t]
    \centering
    \caption{Token variability across prompts with identical macro-F1 and few-shot count, computed per repository and model. Native token counts (left) and \texttt{cl100k}-normalized token counts (right) are reported alongside each other. Savings percentages are consistent across both tokenizers, confirming that the reported variability reflects genuine differences in prompt content.}
    \label{tab:token-range-random-selection-sim_k_sim_fs_sim_f1}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llcc  ccc c  ccc c}
            \toprule
            & & & & \multicolumn{4}{c}{\textbf{Native Tokens}} & \multicolumn{4}{c}{\textbf{cl100k Tokens}} \\
            \cmidrule(lr){5-8} \cmidrule(lr){9-12}
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{\# FS} & \textbf{Min} & \textbf{Max} & $\Delta$ & \textbf{Savings} & \textbf{Min} & \textbf{Max} & $\Delta$ & \textbf{Savings} \\
            \midrule
            \multirow{4}{*}{Bitc

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("../rqs/random_selection")

TARGET_K = 8
F1_COL = "macro_f1"

# models to include
ALLOWED_MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

# repo names for tables
REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

# ============================================================
# STEP 1: LOAD DATA FOR FIXED k
# ============================================================

rows = []

for repo_dir in BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():

        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)

            df.columns = [c.strip() for c in df.columns]

            if not {"FewShot_Count", F1_COL}.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", F1_COL])

            # keep only the target k
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            rows.append({
                "Repository": repo,
                "Model": model_name,
                "F1Values": df[F1_COL].values
            })

# ============================================================
# STEP 2: COMPUTE METRICS PER (REPO, MODEL)
# ============================================================

summary_rows = []

for row in rows:

    f1 = row["F1Values"]

    min_f1 = np.min(f1)
    max_f1 = np.max(f1)
    mean_f1 = np.mean(f1)
    median_f1 = np.median(f1)

    delta_f1 = max_f1 - min_f1

    # relative improvement between best and worst prompt
    rel_improvement = np.inf if min_f1 == 0 else (delta_f1 / min_f1) * 100

    summary_rows.append({

        "Repository": REPO_NAME_MAP.get(row["Repository"], row["Repository"]),
        "Model": row["Model"],

        "min_f1": min_f1,
        "max_f1": max_f1,
        "mean_f1": mean_f1,
        "median_f1": median_f1,

        "delta_f1": delta_f1,
        "rel_impr": rel_improvement,

    })

summary = pd.DataFrame(summary_rows)

# round values for readability
summary = summary.round({
    "min_f1": 2,
    "max_f1": 2,
    "mean_f1": 2,
    "median_f1": 2,
    "delta_f1": 2,
    "rel_impr": 2,
})

summary = summary.sort_values(["Repository", "Model"])

summary

,Repository,Model,min_f1,max_f1,mean_f1,median_f1,delta_f1,rel_impr
1,Bitcoin,Llama-3.1-8B-Instruct,0.36,0.71,0.53,0.53,0.35,95.84
2,Bitcoin,Mistral-7B-Instruct-v0.3,0.63,0.71,0.67,0.67,0.09,13.92
3,Bitcoin,Qwen2.5-7B-Instruct,0.69,0.81,0.77,0.77,0.12,17.05
0,Bitcoin,gemma-3-12b-it,0.73,0.79,0.76,0.76,0.07,9.07
13,OpenCV,Llama-3.1-8B-Instruct,0.17,0.37,0.26,0.25,0.20,116.60
14,OpenCV,Mistral-7B-Instruct-v0.3,0.30,0.49,0.39,0.38,0.20,65.94
15,OpenCV,Qwen2.5-7B-Instruct,0.47,0.71,0.59,0.59,0.24,50.49
12,OpenCV,gemma-3-12b-it,0.57,0.74,0.66,0.66,0.17,29.84
5,React,Llama-3.1-8B-Instruct,0.32,0.67,0.46,0.44,0.34,105.48
6,React,Mistral-7B-Instruct-v0.3,0.46,0.63,0.56,0.57,0.17,36.71


In [10]:
# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    if x == float("inf"):
        return r"$\infty$"
    return f"{x:.2f}"

# ============================================================
# STEP 3: BUILD LATEX TABLE
# ============================================================

latex_rows = []

for repo, repo_df in summary.groupby("Repository", sort=False):

    repo_df = repo_df.reset_index(drop=True)

    for i, r in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
            if i == 0 else ""
        )

        latex_rows.append(
            f"            {repo_cell} & {r['Model']} & "
            f"{fmt_f1(r['min_f1'])} & "
            f"{fmt_f1(r['max_f1'])} & "
            f"{fmt_f1(r['mean_f1'])} & "
            f"{fmt_f1(r['median_f1'])} & "
            f"{fmt_f1(r['delta_f1'])} & "
            f"{r['rel_impr']:.2f}\\% \\\\"
        )

    latex_rows.append("            \\midrule")

latex_rows = latex_rows[:-1]

latex = "\n".join([
    r"\begin{table*}[t]",
    r"    \centering",
    rf"    \caption{{Macro-F1 range across random few-shot selections per repository and model "
    rf"for few-shot count $k = {TARGET_K}$.}}",
    rf"    \label{{tab:f1-range-random-selection-k{TARGET_K}}}",
    r"    \resizebox{\textwidth}{!}{",
    r"        \begin{tabular}{llrrrrrr}",
    r"            \toprule",
    r"            \textbf{Repository} & \textbf{Model} & "
    r"\textbf{Min F1} & \textbf{Max F1} & "
    r"\textbf{Mean F1} & \textbf{Median F1} & "
    r"\textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\",
    r"            \midrule",
    *latex_rows,
    r"            \bottomrule",
    r"        \end{tabular}",
    r"    }",
    r"\end{table*}",
])

# ============================================================
# PRINT CLEAN LATEX
# ============================================================

for line in latex.splitlines():
    print(line)

\begin{table*}[t]
    \centering
    \caption{Macro-F1 range across random few-shot selections per repository and model for few-shot count $k = 8$.}
    \label{tab:f1-range-random-selection-k8}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrrrrrr}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{Min F1} & \textbf{Max F1} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.36 & 0.71 & 0.53 & 0.53 & 0.35 & 95.84\% \\
             & Mistral-7B-Instruct-v0.3 & 0.63 & 0.71 & 0.67 & 0.67 & 0.09 & 13.92\% \\
             & Qwen2.5-7B-Instruct & 0.69 & 0.81 & 0.77 & 0.77 & 0.12 & 17.05\% \\
             & gemma-3-12b-it & 0.73 & 0.79 & 0.76 & 0.76 & 0.07 & 9.07\% \\
            \midrule
            \multirow{4}{*}{OpenCV} & Llama-3.1-8B-Instruct & 0.17 & 0.37 & 0.26 & 0.25 & 0.20 & 116.60\% \\
             & Mistral-7B-Instruct-v0.3 

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("../rqs/similarity_selection_RAG")

TARGET_K = 8

ALLOWED_MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

EMBED_MODEL = "all-mpnet-base-v2"

# ============================================================
# LOAD SIMILARITY RESULTS
# ============================================================

rows = []

for repo_dir in BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / EMBED_MODEL / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():

        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "similarity_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"] = pd.to_numeric(df["Macro_F1"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])

            # filter k
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            # only one row expected
            value = df["Macro_F1"].values[0]

            rows.append({
                "Repository": REPO_NAME_MAP.get(repo, repo),
                "Model": model_name,
                "similarity_f1": value
            })

# ============================================================
# BUILD TABLE
# ============================================================

summary = pd.DataFrame(rows)

summary = summary.round({
    "similarity_f1": 2
})

summary = summary.sort_values(["Repository", "Model"])

In [2]:
summary

,Repository,Model,similarity_f1
1,Bitcoin,Llama-3.1-8B-Instruct,0.58
2,Bitcoin,Mistral-7B-Instruct-v0.3,0.68
3,Bitcoin,Qwen2.5-7B-Instruct,0.80
0,Bitcoin,gemma-3-12b-it,0.79
13,OpenCV,Llama-3.1-8B-Instruct,0.25
14,OpenCV,Mistral-7B-Instruct-v0.3,0.52
15,OpenCV,Qwen2.5-7B-Instruct,0.70
12,OpenCV,gemma-3-12b-it,0.68
5,React,Llama-3.1-8B-Instruct,0.45
6,React,Mistral-7B-Instruct-v0.3,0.53


In [3]:
# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    if x == float("inf"):
        return r"$\infty$"
    return f"{x:.2f}"

# ============================================================
# BUILD LATEX TABLE
# ============================================================

latex_rows = []

for repo, repo_df in summary.groupby("Repository", sort=False):

    repo_df = repo_df.reset_index(drop=True)

    for i, r in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
            if i == 0 else ""
        )

        latex_rows.append(
            f"            {repo_cell} & {r['Model']} & "
            f"{fmt_f1(r['similarity_f1'])} \\\\"
        )

    latex_rows.append("            \\midrule")

# remove last midrule
latex_rows = latex_rows[:-1]

# ============================================================
# FINAL LATEX
# ============================================================

latex = "\n".join([
    r"\begin{table}[t]",
    r"    \centering",
    r"    \caption{Macro-F1 results for similarity-based few-shot selection.}",
    r"    \label{tab:similarity-f1}",
    r"    \begin{tabular}{lll}",
    r"        \toprule",
    r"        \textbf{Repository} & \textbf{Model} & \textbf{Macro F1} \\",
    r"        \midrule",
    *latex_rows,
    r"        \bottomrule",
    r"    \end{tabular}",
    r"\end{table}",
])

# ============================================================
# PRINT CLEAN LATEX
# ============================================================

for line in latex.splitlines():
    print(line)

\begin{table}[t]
    \centering
    \caption{Macro-F1 results for similarity-based few-shot selection.}
    \label{tab:similarity-f1}
    \begin{tabular}{lll}
        \toprule
        \textbf{Repository} & \textbf{Model} & \textbf{Macro F1} \\
        \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.58 \\
             & Mistral-7B-Instruct-v0.3 & 0.68 \\
             & Qwen2.5-7B-Instruct & 0.80 \\
             & gemma-3-12b-it & 0.79 \\
            \midrule
            \multirow{4}{*}{OpenCV} & Llama-3.1-8B-Instruct & 0.25 \\
             & Mistral-7B-Instruct-v0.3 & 0.52 \\
             & Qwen2.5-7B-Instruct & 0.70 \\
             & gemma-3-12b-it & 0.68 \\
            \midrule
            \multirow{4}{*}{React} & Llama-3.1-8B-Instruct & 0.45 \\
             & Mistral-7B-Instruct-v0.3 & 0.53 \\
             & Qwen2.5-7B-Instruct & 0.66 \\
             & gemma-3-12b-it & 0.75 \\
            \midrule
            \multirow{4}{*}{Tensorflow} & Llama-3.1-8B-Instru

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

RANDOM_BASE_DIR = Path("../rqs/random_selection")
SIM_BASE_DIR = Path("../rqs/similarity_selection_RAG")

TARGET_K = 8

ALLOWED_MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

EMBED_MODEL = "all-mpnet-base-v2"

# ============================================================
# LOAD RANDOM SELECTION (k = 8)
# ============================================================

random_rows = []

for repo_dir in RANDOM_BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required = {"FewShot_Count", "Macro_F1"}
            if not required.issubset(df.columns):
                print("Skipping (missing columns):", csv_path)
                continue

            # optional token column
            has_token = "Token_Count" in df.columns

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"] = pd.to_numeric(df["Macro_F1"], errors="coerce")

            if has_token:
                df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")
            else:
                df["Token_Count"] = np.nan

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            f1_values = df["Macro_F1"].values

            min_f1 = np.min(f1_values)
            max_f1 = np.max(f1_values)
            mean_f1 = np.mean(f1_values)
            median_f1 = np.median(f1_values)

            delta_f1 = max_f1 - min_f1
            rel_impr = np.inf if min_f1 == 0 else (delta_f1 / min_f1) * 100

            # token for best run
            max_row = df.loc[df["Macro_F1"].idxmax()]
            max_token = max_row["Token_Count"]

            random_rows.append({
                "Repository": REPO_NAME_MAP.get(repo, repo),
                "Model": model_name,

                "min_f1": min_f1,
                "max_f1": max_f1,
                "mean_f1": mean_f1,
                "median_f1": median_f1,
                "delta_f1": delta_f1,
                "rel_impr": rel_impr,

                "max_token": max_token
            })

random_df = pd.DataFrame(random_rows)

# ============================================================
# LOAD SIMILARITY SELECTION (k = 8)
# ============================================================

sim_rows = []

for repo_dir in SIM_BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / EMBED_MODEL / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "similarity_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required = {"FewShot_Count", "Macro_F1"}
            if not required.issubset(df.columns):
                print("Skipping (missing columns):", csv_path)
                continue

            has_token = "Token_Count" in df.columns

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"] = pd.to_numeric(df["Macro_F1"], errors="coerce")

            if has_token:
                df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")
            else:
                df["Token_Count"] = np.nan

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            sim_rows.append({
                "Repository": REPO_NAME_MAP.get(repo, repo),
                "Model": model_name,
                "sim_f1": df["Macro_F1"].values[0],
                "sim_token": df["Token_Count"].values[0]
            })

sim_df = pd.DataFrame(sim_rows)

# ============================================================
# MERGE
# ============================================================

if random_df.empty:
    raise ValueError("random_df is empty - check your random CSV columns")

combined = pd.merge(
    sim_df,
    random_df,
    on=["Repository", "Model"],
    how="inner"
)

combined = combined.sort_values(["Repository", "Model"])

# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    return r"$\infty$" if x == float("inf") else f"{x:.2f}"

def fmt_percent(x):
    return r"$\infty$" if x == float("inf") else f"{x:.1f}\\%"

def fmt_int(x):
    return "-" if pd.isna(x) else f"{int(x)}"

# ============================================================
# LATEX TABLE
# ============================================================

print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Similarity vs Random few-shot selection at $k=8$ (including token usage).}")
print("    \\label{tab:sim-vs-random}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llrr|rrrrrrr}")
print("            \\toprule")

print("            & & \\multicolumn{2}{c}{\\textbf{Similarity}} "
      "& \\multicolumn{7}{c}{\\textbf{Random}} \\\\")

print("            \\textbf{Repository} & \\textbf{Model} "
      "& \\textbf{F1} & \\textbf{Tokens} "
      "& \\textbf{Min} & \\textbf{Max} & \\textbf{Mean} & \\textbf{Median} "
      "& \\textbf{$\\Delta$} & \\textbf{Rel. (\\%)} & \\textbf{Max Tokens} \\\\")

print("            \\midrule")

repos = combined["Repository"].unique()

for i, repo in enumerate(repos):

    repo_df = combined[combined["Repository"] == repo]
    n_models = len(repo_df)

    for j, (_, r) in enumerate(repo_df.iterrows()):

        repo_cell = f"\\multirow{{{n_models}}}{{*}}{{{repo}}}" if j == 0 else ""

        print(
            f"            {repo_cell} & {r['Model']} "
            f"& {fmt_f1(r['sim_f1'])} & {fmt_int(r['sim_token'])} "
            f"& {fmt_f1(r['min_f1'])} & {fmt_f1(r['max_f1'])} "
            f"& {fmt_f1(r['mean_f1'])} & {fmt_f1(r['median_f1'])} "
            f"& {fmt_f1(r['delta_f1'])} & {fmt_percent(r['rel_impr'])} "
            f"& {fmt_int(r['max_token'])} \\\\"
        )

    if i < len(repos) - 1:
        print("            \\midrule")

print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")

\begin{table*}[t]
    \centering
    \caption{Similarity vs Random few-shot selection at $k=8$ (including token usage).}
    \label{tab:sim-vs-random}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrr|rrrrrrr}
            \toprule
            & & \multicolumn{2}{c}{\textbf{Similarity}} & \multicolumn{7}{c}{\textbf{Random}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{Tokens} & \textbf{Min} & \textbf{Max} & \textbf{Mean} & \textbf{Median} & \textbf{$\Delta$} & \textbf{Rel. (\%)} & \textbf{Max Tokens} \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.58 & 2881 & 0.36 & 0.71 & 0.53 & 0.53 & 0.35 & 95.8\% & 2613 \\
             & Mistral-7B-Instruct-v0.3 & 0.68 & 3502 & 0.63 & 0.71 & 0.67 & 0.67 & 0.09 & 13.9\% & 7518 \\
             & Qwen2.5-7B-Instruct & 0.80 & 3122 & 0.69 & 0.81 & 0.77 & 0.77 & 0.12 & 17.0\% & 3426 \\
             & gemma-3-12b-it & 0.79 & 3243 & 0.73 & 0.79 & 0.76 & 0.76 & 0.07 & 9.1\% & 1

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

RANDOM_BASE_DIR    = Path("../rqs/random_selection")
SIM_BASE_DIR       = Path("../rqs/similarity_selection_RAG")
RANDOM_CL100K_BASE = Path("../rqs/random_selection")

TARGET_K = 8

ALLOWED_MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

REPO_NAME_MAP = {
    "facebook_react":        "React",
    "bitcoin_bitcoin":       "Bitcoin",
    "opencv_opencv":         "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode":      "VsCode",
}

EMBED_MODEL = "all-mpnet-base-v2"

# ============================================================
# LOAD RANDOM SELECTION
# ============================================================

random_rows = []

for repo_dir in RANDOM_BASE_DIR.iterdir():
    if not repo_dir.is_dir():
        continue
    repo = repo_dir.name
    models_dir = repo_dir / "models"
    if not models_dir.exists():
        continue

    # Load cl100k for this repo (model-agnostic)
    cl100k_file = RANDOM_CL100K_BASE / repo / "cl100k_token_counts.json"
    cl100k_data = None
    if cl100k_file.exists():
        with open(cl100k_file, "r") as f:
            cl100k_data = json.load(f)

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue
        for model_dir in provider_dir.iterdir():
            model_name = model_dir.name
            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"
            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            if not {"FewShot_Count", "Macro_F1"}.issubset(df.columns):
                print(f"Skipping (missing columns): {csv_path}")
                continue

            has_token = "Token_Count" in df.columns
            has_trial = "Trial" in df.columns

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"]      = pd.to_numeric(df["Macro_F1"],      errors="coerce")
            if has_token:
                df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")
            if has_trial:
                df["Trial"] = pd.to_numeric(df["Trial"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])
            df = df[df["FewShot_Count"] == TARGET_K]
            if df.empty:
                continue

            f1_values = df["Macro_F1"].values
            best_row  = df.loc[df["Macro_F1"].idxmax()]

            # Native token of best trial
            best_native_token = best_row["Token_Count"] if has_token else np.nan

            # cl100k token of best trial (matched by Trial index)
            best_cl100k = np.nan
            if cl100k_data is not None and has_trial and str(TARGET_K) in cl100k_data:
                best_trial_idx = int(best_row["Trial"])
                match = next(
                    (t for t in cl100k_data[str(TARGET_K)]
                     if t["trial"] == best_trial_idx),
                    None
                )
                if match:
                    best_cl100k = match["cl100k_tokens"]
                else:
                    print(f"  [MISSING] trial {best_trial_idx} not found "
                          f"in cl100k json for {repo} k={TARGET_K}")

            random_rows.append({
                "Repository":   REPO_NAME_MAP.get(repo, repo),
                "Model":        model_name,
                "min_f1":       np.min(f1_values),
                "max_f1":       np.max(f1_values),
                "mean_f1":      np.mean(f1_values),
                "median_f1":    np.median(f1_values),
                "delta_f1":     np.max(f1_values) - np.min(f1_values),
                "rel_impr":     np.inf if np.min(f1_values) == 0
                                else ((np.max(f1_values) - np.min(f1_values))
                                      / np.min(f1_values)) * 100,
                "best_token":       best_native_token,
                "best_cl100k":      best_cl100k,
            })

random_df = pd.DataFrame(random_rows)

# ============================================================
# LOAD SIMILARITY SELECTION
# ============================================================

sim_rows = []

for repo_dir in SIM_BASE_DIR.iterdir():
    if not repo_dir.is_dir():
        continue
    repo = repo_dir.name
    models_dir = repo_dir / EMBED_MODEL / "models"
    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue
        for model_dir in provider_dir.iterdir():
            model_name = model_dir.name
            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "similarity_results_macro_f1.csv"
            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            if not {"FewShot_Count", "Macro_F1"}.issubset(df.columns):
                print(f"Skipping (missing columns): {csv_path}")
                continue

            has_native = "Token_Count" in df.columns
            has_cl100k = "Token_Count_cl100k" in df.columns

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"]      = pd.to_numeric(df["Macro_F1"],      errors="coerce")
            if has_native:
                df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")
            if has_cl100k:
                df["Token_Count_cl100k"] = pd.to_numeric(
                    df["Token_Count_cl100k"], errors="coerce"
                )

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])
            df = df[df["FewShot_Count"] == TARGET_K]
            if df.empty:
                continue

            # Read cl100k directly from json (most reliable)
            cl100k_file = (
                SIM_BASE_DIR / repo_dir.name / EMBED_MODEL
                / f"{provider_dir.name}_{model_name}"
                / "cl100k_token_counts.json"
            )
            sim_cl100k = np.nan
            if cl100k_file.exists():
                with open(cl100k_file, "r") as f:
                    cl100k_json = json.load(f)
                if str(TARGET_K) in cl100k_json:
                    sim_cl100k = cl100k_json[str(TARGET_K)]["mean_cl100k_tokens"]
            else:
                print(f"  [MISSING cl100k] {cl100k_file}")

            sim_rows.append({
                "Repository": REPO_NAME_MAP.get(repo_dir.name, repo_dir.name),
                "Model":      model_name,
                "sim_f1":     df["Macro_F1"].values[0],
                "sim_token":  df["Token_Count"].values[0] if has_native else np.nan,
                "sim_cl100k": sim_cl100k,
            })

sim_df = pd.DataFrame(sim_rows)

# ============================================================
# MERGE
# ============================================================

if random_df.empty:
    raise ValueError("random_df is empty - check your CSV columns")

combined = pd.merge(sim_df, random_df, on=["Repository", "Model"], how="inner")
combined = combined.sort_values(["Repository", "Model"])

# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    return r"$\infty$" if x == float("inf") else f"{x:.2f}"

def fmt_percent(x):
    return r"$\infty$" if x == float("inf") else f"{x:.1f}\\%"

def fmt_int(x):
    return "-" if pd.isna(x) else f"{int(round(x))}"

def random_higher(sim_val, rand_val):
    if pd.isna(sim_val) or pd.isna(rand_val):
        return "-"
    return "\\cmark" if rand_val > sim_val else "\\xmark"

# ============================================================
# LATEX — PRIMARY TABLE (native tokens)
# ============================================================

def print_native_table(combined):
    repos    = combined["Repository"].unique()
    print("\\begin{table*}[t]")
    print("    \\centering")
    print("    \\caption{Similarity vs.\ Random few-shot selection at $k=8$ "
          "(native token usage). For similarity-based selection, token counts "
          "reflect the mean across all test instances. For random selection, "
          "token counts reflect the prompt length of the best-performing trial.}")
    print("    \\label{tab:sim-vs-random}")
    print("    \\resizebox{\\textwidth}{!}{")
    print("        \\begin{tabular}{llrr|rrrrrrr}")
    print("            \\toprule")
    print("            & & \\multicolumn{2}{c|}{\\textbf{Similarity}} "
          "& \\multicolumn{7}{c}{\\textbf{Random}} \\\\")
    print("            \\textbf{Repository} & \\textbf{Model} "
          "& \\textbf{F1} & \\textbf{Tokens} "
          "& \\textbf{Min} & \\textbf{Max} & \\textbf{Mean} & \\textbf{Median} "
          "& \\textbf{$\\Delta$} & \\textbf{Rel. (\\%)} & \\textbf{Best Tokens} \\\\")
    print("            \\midrule")

    for i, repo in enumerate(repos):
        repo_df = combined[combined["Repository"] == repo]
        for j, (_, r) in enumerate(repo_df.iterrows()):
            repo_cell = (f"\\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
                         if j == 0 else "")
            print(
                f"            {repo_cell} & {r['Model']} "
                f"& {fmt_f1(r['sim_f1'])} & {fmt_int(r['sim_token'])} "
                f"& {fmt_f1(r['min_f1'])} & {fmt_f1(r['max_f1'])} "
                f"& {fmt_f1(r['mean_f1'])} & {fmt_f1(r['median_f1'])} "
                f"& {fmt_f1(r['delta_f1'])} & {fmt_percent(r['rel_impr'])} "
                f"& {fmt_int(r['best_token'])} \\\\"
            )
        if i < len(repos) - 1:
            print("            \\midrule")

    print("            \\bottomrule")
    print("        \\end{tabular}}")
    print("\\end{table*}")

# ============================================================
# LATEX — COMPANION TABLE (cl100k tokens)
# ============================================================

def print_cl100k_table(combined):
    repos = combined["Repository"].unique()

    # Count how many cases random > sim
    n_random_higher = sum(
        1 for _, r in combined.iterrows()
        if not pd.isna(r["best_cl100k"]) and not pd.isna(r["sim_cl100k"])
        and r["best_cl100k"] > r["sim_cl100k"]
    )
    total = len(combined.dropna(subset=["best_cl100k", "sim_cl100k"]))
    print(f"% Random > Sim under cl100k: {n_random_higher}/{total}")

    print("\\begin{table}[t]")
    print("    \\centering")
    print("    \\caption{Similarity vs.\ Random few-shot selection at $k=8$: "
          "\\texttt{cl100k}-normalized token usage. Companion to "
          "Table~\\ref{tab:sim-vs-random}. For random selection, the token "
          "count of the best-performing trial is reported. For similarity "
          "selection, the mean across all test instances is reported.}")
    print("    \\label{tab:sim-vs-random-cl100k}")
    print("    \\footnotesize")
    print("    \\begin{tabular}{llr|rr|c}")
    print("        \\toprule")
    print("        & & \\multicolumn{1}{c|}{\\textbf{Similarity}} "
          "& \\multicolumn{2}{c|}{\\textbf{Random}} & \\\\")
    print("        \\textbf{Repo} & \\textbf{Model} "
          "& \\textbf{cl100k (mean)} "
          "& \\textbf{F1} & \\textbf{cl100k (best)} "
          "& \\textbf{Rnd $>$ Sim?} \\\\")
    print("        \\midrule")

    for i, repo in enumerate(repos):
        repo_df = combined[combined["Repository"] == repo]
        for j, (_, r) in enumerate(repo_df.iterrows()):
            repo_cell = (f"\\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}"
                         if j == 0 else "")
            print(
                f"        {repo_cell} & {r['Model']} "
                f"& {fmt_int(r['sim_cl100k'])} "
                f"& {fmt_f1(r['max_f1'])} & {fmt_int(r['best_cl100k'])} "
                f"& {random_higher(r['sim_cl100k'], r['best_cl100k'])} \\\\"
            )
        if i < len(repos) - 1:
            print("        \\midrule")

    print("        \\bottomrule")
    print("    \\end{tabular}")
    print("\\end{table}")

# ============================================================
# PRINT BOTH TABLES
# ============================================================

print("% ============================================================")
print("% PRIMARY TABLE — native tokens")
print("% ============================================================\n")
print_native_table(combined)

print("\n\n% ============================================================")
print("% COMPANION TABLE — cl100k normalized tokens")
print("% ============================================================\n")
print_cl100k_table(combined)

<>:237: SyntaxWarning: invalid escape sequence '\ '
<>:291: SyntaxWarning: invalid escape sequence '\ '
<>:237: SyntaxWarning: invalid escape sequence '\ '
<>:291: SyntaxWarning: invalid escape sequence '\ '
C:\Users\AU61310\AppData\Local\Temp\ipykernel_57432\2711370770.py:237: SyntaxWarning: invalid escape sequence '\ '
  print("    \\caption{Similarity vs.\ Random few-shot selection at $k=8$ "
C:\Users\AU61310\AppData\Local\Temp\ipykernel_57432\2711370770.py:291: SyntaxWarning: invalid escape sequence '\ '
  print("    \\caption{Similarity vs.\ Random few-shot selection at $k=8$: "


% ============================================================
% PRIMARY TABLE — native tokens
% ============================================================

\begin{table*}[t]
    \centering
    \caption{Similarity vs.\ Random few-shot selection at $k=8$ (native token usage). For similarity-based selection, token counts reflect the mean across all test instances. For random selection, token counts reflect the prompt length of the best-performing trial.}
    \label{tab:sim-vs-random}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrr|rrrrrrr}
            \toprule
            & & \multicolumn{2}{c|}{\textbf{Similarity}} & \multicolumn{7}{c}{\textbf{Random}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{Tokens} & \textbf{Min} & \textbf{Max} & \textbf{Mean} & \textbf{Median} & \textbf{$\Delta$} & \textbf{Rel. (\%)} & \textbf{Best Tokens} \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.58 & 7362 & 0.36 & 0.71 &

In [6]:
# ============================================================
# LATEX — COMBINED TABLE (native + cl100k)
# ============================================================

def print_combined_table(combined):
    repos = combined["Repository"].unique()
    
    print("\\begin{table*}[t]")
    print("    \\centering")
    print("    \\caption{Similarity vs. Random few-shot selection at $k=8$. "
          "Similarity columns show macro-F1 and token usage. Random columns show "
          "min, max, mean, median, $\Delta$, relative improvement, and best token counts "
          "(both native and cl100k).}")
    print("    \\label{tab:sim-vs-random-combined}")
    print("    \\resizebox{\\textwidth}{!}{")
    print("        \\begin{tabular}{llrrr|rrrrrrrr}")
    print("            \\toprule")
    print("            & & \\multicolumn{3}{c|}{\\textbf{Similarity}} "
          "& \\multicolumn{8}{c}{\\textbf{Random}} \\\\")
    print("            \\textbf{Repository} & \\textbf{Model} "
          "& \\textbf{F1} & \\textbf{Tokens (native)} & \\textbf{Tokens (cl100k)} "
          "& \\textbf{Min F1} & \\textbf{Max F1} & \\textbf{Mean F1} & \\textbf{Median F1} "
          "& \\textbf{$\\Delta$} & \\textbf{Rel. (\%)} & \\textbf{Best Tokens (native)} "
          "& \\textbf{Best Tokens (cl100k)} \\\\")
    print("            \\midrule")

    for i, repo in enumerate(repos):
        repo_df = combined[combined["Repository"] == repo]
        for j, (_, r) in enumerate(repo_df.iterrows()):
            repo_cell = f"\\multirow{{{len(repo_df)}}}{{*}}{{{repo}}}" if j == 0 else ""
            print(
                f"            {repo_cell} & {r['Model']} "
                f"& {fmt_f1(r['sim_f1'])} & {fmt_int(r['sim_token'])} & {fmt_int(r['sim_cl100k'])} "
                f"& {fmt_f1(r['min_f1'])} & {fmt_f1(r['max_f1'])} & {fmt_f1(r['mean_f1'])} "
                f"& {fmt_f1(r['median_f1'])} & {fmt_f1(r['delta_f1'])} & {fmt_percent(r['rel_impr'])} "
                f"& {fmt_int(r['best_token'])} & {fmt_int(r['best_cl100k'])} \\\\"
            )
        if i < len(repos) - 1:
            print("            \\midrule")

    print("            \\bottomrule")
    print("        \\end{tabular}}")
    print("\\end{table*}")


# ============================================================
# PRINT COMBINED TABLE
# ============================================================

print("% ============================================================")
print("% COMBINED TABLE — similarity + random + native + cl100k tokens")
print("% ============================================================\n")
print_combined_table(combined)

% ============================================================
% COMBINED TABLE — similarity + random + native + cl100k tokens
% ============================================================

\begin{table*}[t]
    \centering
    \caption{Similarity vs. Random few-shot selection at $k=8$. Similarity columns show macro-F1 and token usage. Random columns show min, max, mean, median, $\Delta$, relative improvement, and best token counts (both native and cl100k).}
    \label{tab:sim-vs-random-combined}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrrr|rrrrrrrr}
            \toprule
            & & \multicolumn{3}{c|}{\textbf{Similarity}} & \multicolumn{8}{c}{\textbf{Random}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{Tokens (native)} & \textbf{Tokens (cl100k)} & \textbf{Min F1} & \textbf{Max F1} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$} & \textbf{Rel. (\%)} & \textbf{Best Tokens (native)} & \textbf{Best Tokens (cl100k)} \\
       

<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:23: SyntaxWarning: invalid escape sequence '\%'
<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:23: SyntaxWarning: invalid escape sequence '\%'
C:\Users\AU61310\AppData\Local\Temp\ipykernel_57432\1876977853.py:12: SyntaxWarning: invalid escape sequence '\D'
  "min, max, mean, median, $\Delta$, relative improvement, and best token counts "
C:\Users\AU61310\AppData\Local\Temp\ipykernel_57432\1876977853.py:23: SyntaxWarning: invalid escape sequence '\%'
  "& \\textbf{$\\Delta$} & \\textbf{Rel. (\%)} & \\textbf{Best Tokens (native)} "
